In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
# Load all CSV files
processed_path = Path('../data/processed/')
processed_files = list(processed_path.glob('*.csv'))

if not processed_files:
    print(f"\n❌ No CSV files found in {processed_path}")
else:
    print(f"\n📁 Found {len(processed_files)} CSV files in {processed_path}")

In [ ]:
# Check each files
results = []
detailed_issues = {}

for filepath in sorted(processed_files):
    table_name = filepath.stem
    
    print(f"\n📋 Checking: {table_name.upper()}")
    print("-" * 50)
    
    try:
        # Load the CSV
        df = pd.read_csv(filepath)
        
        # Basic info
        print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        # Check for missing values
        missing_counts = df.isnull().sum()
        missing_cols = missing_counts[missing_counts > 0]
        total_missing = missing_counts.sum()
        
        if total_missing > 0:
            print(f"   ⚠️ Missing Values: {total_missing:,}")
            for col, count in missing_cols.items():
                pct = (count / len(df)) * 100
                print(f"      - {col}: {count:,} ({pct:.1f}%)")
        else:
            print(f"   ✅ No missing values")
        
        # Check for duplicates
        duplicates = df.duplicated().sum()
        if duplicates > 0:
            print(f"   ⚠️ Duplicates: {duplicates:,} ({duplicates/len(df)*100:.1f}%)")
        else:
            print(f"   ✅ No duplicates")
        
        # Check for outliers in numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        outlier_counts = {}
        
        for col in numeric_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            
            if IQR != 0:
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
                outlier_count = outlier_mask.sum()
                
                if outlier_count > 0:
                    outlier_counts[col] = {
                        'count': outlier_count,
                        'percentage': (outlier_count / len(df)) * 100,
                        'lower_bound': lower_bound,
                        'upper_bound': upper_bound,
                        'min': df[col].min(),
                        'max': df[col].max()
                    }
        
        if outlier_counts:
            print(f"   ⚠️ Outliers detected in {len(outlier_counts)} columns:")
            for col, info in outlier_counts.items():
                print(f"      - {col}: {info['count']:,} ({info['percentage']:.1f}%)")
                print(f"        Range: {info['min']:.2f} - {info['max']:.2f}")
                print(f"        Bounds: {info['lower_bound']:.2f} - {info['upper_bound']:.2f}")
        else:
            print(f"   ✅ No outliers detected")
        
        # Store results
        results.append({
            'Table': table_name,
            'Rows': df.shape[0],
            'Columns': df.shape[1],
            'Missing_Count': total_missing,
            'Missing_Columns': len(missing_cols),
            'Duplicates': duplicates,
            'Outlier_Columns': len(outlier_counts),
            'Status': '✅' if (total_missing == 0 and duplicates == 0 and len(outlier_counts) == 0) else '⚠️'
        })
        
        if total_missing > 0 or duplicates > 0 or outlier_counts:
            detailed_issues[table_name] = {
                'missing': missing_cols.to_dict() if total_missing > 0 else {},
                'duplicates': duplicates,
                'outliers': {col: info['count'] for col, info in outlier_counts.items()}
            }
        
        print("-" * 50)
        
    except Exception as e:
        print(f"   ❌ Error loading {filepath.name}: {e}")
        results.append({
            'Table': table_name,
            'Rows': 0,
            'Columns': 0,
            'Missing_Count': 0,
            'Missing_Columns': 0,
            'Duplicates': 0,
            'Outlier_Columns': 0,
            'Status': '❌ Error'
        })

# Display summary

print("\n" + "="*80)
print("📊 QUALITY CHECK SUMMARY")
print("="*80)

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

# ============================================================
# 4. CHECK FOR CLEAN DATA
# ============================================================

print("\n" + "="*80)
print("✅ CLEAN DATA CHECK")
print("="*80)

clean_tables = [r['Table'] for r in results if r['Status'] == '✅']
issue_tables = [r['Table'] for r in results if r['Status'] == '⚠️']
error_tables = [r['Table'] for r in results if r['Status'] == '❌ Error']

if clean_tables:
    print(f"\n✅ Clean tables ({len(clean_tables)}):")
    for table in clean_tables:
        print(f"   ✓ {table}")

if issue_tables:
    print(f"\n⚠️ Tables with issues ({len(issue_tables)}):")
    for table in issue_tables:
        print(f"   ⚠️ {table}")
        
    # Show detailed issues
    print("\n" + "-" * 50)
    print("DETAILED ISSUES:")
    for table_name, issues in detailed_issues.items():
        print(f"\n📋 {table_name.upper()}:")
        
        if issues['missing']:
            print(f"   Missing values:")
            for col, count in issues['missing'].items():
                print(f"      - {col}: {count}")
        
        if issues['duplicates'] > 0:
            print(f"   Duplicates: {issues['duplicates']:,}")
        
        if issues['outliers']:
            print(f"   Outliers:")
            for col, count in issues['outliers'].items():
                print(f"      - {col}: {count:,}")

if error_tables:
    print(f"\n❌ Tables with errors ({len(error_tables)}):")
    for table in error_tables:
        print(f"   ❌ {table}")

# Save Reports

print("\n" + "="*80)
print("💾 SAVING QUALITY REPORT")
print("="*80)

# Create a detailed report
report_path = Path('../reports/')
report_path.mkdir(parents=True, exist_ok=True)

# Save summary as CSV
summary_df.to_csv(report_path / 'quality_check_summary.csv', index=False)
print(f"   ✓ Saved summary to {report_path / 'quality_check_summary.csv'}")

# Create a detailed text report
report_file = report_path / 'quality_check_report.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("DATA QUALITY CHECK REPORT\n")
    f.write("="*80 + "\n\n")
    
    for result in results:
        f.write(f"\n{result['Table'].upper()}:\n")
        f.write(f"   Rows: {result['Rows']:,}\n")
        f.write(f"   Columns: {result['Columns']}\n")
        f.write(f"   Missing Values: {result['Missing_Count']:,}\n")
        f.write(f"   Duplicates: {result['Duplicates']:,}\n")
        f.write(f"   Outlier Columns: {result['Outlier_Columns']}\n")
        f.write(f"   Status: {result['Status']}\n")
        
        # Add detailed issues if any
        if result['Table'] in detailed_issues:
            issues = detailed_issues[result['Table']]
            if issues['missing']:
                f.write(f"   Missing Details:\n")
                for col, count in issues['missing'].items():
                    f.write(f"      - {col}: {count}\n")
            if issues['duplicates'] > 0:
                f.write(f"   Duplicates: {issues['duplicates']:,}\n")
            if issues['outliers']:
                f.write(f"   Outlier Details:\n")
                for col, count in issues['outliers'].items():
                    f.write(f"      - {col}: {count:,}\n")
        f.write("-"*40 + "\n")
    
    # Overall summary
    f.write("\n" + "="*80 + "\n")
    f.write("OVERALL SUMMARY\n")
    f.write("="*80 + "\n")
    f.write(f"Clean Tables: {len(clean_tables)}\n")
    f.write(f"Tables with Issues: {len(issue_tables)}\n")
    f.write(f"Tables with Errors: {len(error_tables)}\n")
    f.write("="*80 + "\n")

print(f"   ✓ Saved detailed report to {report_file}")

# Check Status 

print("\n" + "="*80)
print("🎯 FINAL STATUS")
print("="*80)

if not issue_tables and not error_tables:
    print("\n🎉 ALL TABLES ARE CLEAN! No missing values, duplicates, or outliers found.")
elif not error_tables:
    print(f"\n⚠️ {len(issue_tables)} tables have issues that need attention.")
    print("   Check the detailed report above for specifics.")
else:
    print(f"\n❌ {len(error_tables)} tables failed to load. Check file permissions.")

print("\n" + "="*80)
print("✅ QUALITY CHECK COMPLETED!")
print("="*80)


📁 Found 11 CSV files in ..\data\processed

📋 Checking: ACCOUNT_STATUSES
--------------------------------------------------
   Shape: 3 rows × 2 columns
   Memory: 0.00 MB
   ✅ No missing values
   ✅ No duplicates
   ✅ No outliers detected
--------------------------------------------------

📋 Checking: ACCOUNT_TYPES
--------------------------------------------------
   Shape: 5 rows × 2 columns
   Memory: 0.00 MB
   ✅ No missing values
   ✅ No duplicates
   ✅ No outliers detected
--------------------------------------------------

📋 Checking: ACCOUNTS
--------------------------------------------------
   Shape: 1,651 rows × 6 columns
   Memory: 0.18 MB
   ✅ No missing values
   ✅ No duplicates
   ✅ No outliers detected
--------------------------------------------------

📋 Checking: ADDRESSES
--------------------------------------------------
   Shape: 1,210 rows × 4 columns
   Memory: 0.21 MB
   ✅ No missing values
   ✅ No duplicates
   ✅ No outliers detected
--------------------------

In [3]:
print("\n📊 PROCESSED DATA SUMMARY")
print("="*70)

for filepath in sorted(Path('../data/processed/').glob('*.csv')):
    df = pd.read_csv(filepath)
    table_name = filepath.stem
    
    print(f"\n✅ {table_name:20s} → {df.shape[0]:>6,} rows × {df.shape[1]:>3} cols")
    print(f"   Columns: {', '.join(df.columns.tolist())}")
    print("-" * 70)

print("="*70)


📊 PROCESSED DATA SUMMARY

✅ account_statuses     →      3 rows ×   2 cols
   Columns: AccountStatusID, StatusName
----------------------------------------------------------------------

✅ account_types        →      5 rows ×   2 cols
   Columns: AccountTypeID, TypeName
----------------------------------------------------------------------

✅ accounts             →  1,651 rows ×   6 cols
   Columns: AccountID, CustomerID, AccountTypeID, AccountStatusID, Balance, OpeningDate
----------------------------------------------------------------------

✅ addresses            →  1,210 rows ×   4 cols
   Columns: AddressID, Street, City, Country
----------------------------------------------------------------------

✅ branches             →     50 rows ×   3 cols
   Columns: BranchID, BranchName, AddressID
----------------------------------------------------------------------

✅ customer_types       →      3 rows ×   2 cols
   Columns: CustomerTypeID, TypeName
-----------------------------------

In [ ]:


# Load data
data = {f.stem: pd.read_csv(f) for f in Path('../data/processed/').glob('*.csv')}
print(f"\n✅ Loaded {len(data)} tables")


# TRANSACTION-LEVEL DATASET


print("\n📋 Creating Transaction Dataset...")

df_trans = data['transactions'].copy()

# Merge all related tables
for merge_config in [
    ('accounts', 'AccountOriginID', 'AccountID', '_origin', {'CustomerID': 'OriginCustomerID', 'AccountTypeID': 'OriginAccountTypeID', 'AccountStatusID': 'OriginAccountStatusID', 'Balance': 'OriginBalance', 'OpeningDate': 'OriginOpeningDate'}),
    ('accounts', 'AccountDestinationID', 'AccountID', '_dest', {'CustomerID': 'DestCustomerID', 'AccountTypeID': 'DestAccountTypeID', 'AccountStatusID': 'DestAccountStatusID', 'Balance': 'DestBalance', 'OpeningDate': 'DestOpeningDate'}),
    ('customers', 'OriginCustomerID', 'CustomerID', '', {'FirstName': 'OriginFirstName', 'LastName': 'OriginLastName', 'DateOfBirth': 'OriginDateOfBirth', 'AddressID': 'OriginAddressID', 'CustomerTypeID': 'OriginCustomerTypeID'}),
    ('customers', 'DestCustomerID', 'CustomerID', '_dest', {'FirstName': 'DestFirstName', 'LastName': 'DestLastName', 'DateOfBirth': 'DestDateOfBirth', 'AddressID': 'DestAddressID', 'CustomerTypeID': 'DestCustomerTypeID'}),
    ('transaction_types', 'TransactionTypeID', 'TransactionTypeID', ''),
    ('branches', 'BranchID', 'BranchID', '')
]:
    table, left_on, right_on, suffix, rename = merge_config[0], merge_config[1], merge_config[2], merge_config[3] if len(merge_config) > 3 else '', merge_config[4] if len(merge_config) > 4 else {}
    df_trans = df_trans.merge(data[table], left_on=left_on, right_on=right_on, how='left', suffixes=('', suffix))
    if rename:
        df_trans.rename(columns=rename, inplace=True)

# Cleanup duplicate columns
df_trans.drop(columns=[c for c in df_trans.columns if c.endswith('_y')], errors='ignore', inplace=True)
df_trans.drop(columns=['AccountID_x', 'AccountID_y', 'CustomerID_x', 'CustomerID_y'], errors='ignore', inplace=True)

print(f"   ✅ {df_trans.shape[0]:,} rows × {df_trans.shape[1]} columns")


# ACCOUNT-LEVEL DATASET


print("\n📋 Creating Account Dataset...")

df_acc = data['accounts'].copy()

# Merge with customers, types, statuses
for table, on, rename in [
    ('customers', 'CustomerID', {}),
    ('account_types', 'AccountTypeID', {}),
    ('account_statuses', 'AccountStatusID', {})
]:
    df_acc = df_acc.merge(data[table], on=on, how='left')
    if rename:
        df_acc.rename(columns=rename, inplace=True)

# Add loan summary
loan_summary = data['loans'].groupby('AccountID').agg(
    LoanCount=('LoanID', 'count'),
    TotalPrincipal=('PrincipalAmount', 'sum'),
    AvgPrincipal=('PrincipalAmount', 'mean'),
    MaxPrincipal=('PrincipalAmount', 'max'),
    AvgInterestRate=('InterestRate', 'mean'),
    MaxInterestRate=('InterestRate', 'max')
).reset_index()

for col in ['LoanCount', 'TotalPrincipal', 'AvgPrincipal', 'MaxPrincipal', 'AvgInterestRate', 'MaxInterestRate']:
    loan_summary[col] = loan_summary[col].fillna(0)

df_acc = df_acc.merge(loan_summary, on='AccountID', how='left')

# Add transaction summary
trans_summary = data['transactions'].groupby('AccountOriginID').agg(
    TransactionCount=('TransactionID', 'count'),
    TotalTransactionAmount=('Amount', 'sum'),
    AvgTransactionAmount=('Amount', 'mean'),
    MaxTransactionAmount=('Amount', 'max'),
    MinTransactionAmount=('Amount', 'min')
).reset_index().rename(columns={'AccountOriginID': 'AccountID'})

for col in ['TransactionCount', 'TotalTransactionAmount', 'AvgTransactionAmount', 'MaxTransactionAmount', 'MinTransactionAmount']:
    trans_summary[col] = trans_summary[col].fillna(0)

df_acc = df_acc.merge(trans_summary, on='AccountID', how='left')

print(f"   ✅ {df_acc.shape[0]:,} rows × {df_acc.shape[1]} columns")


# CUSTOMER-LEVEL DATASET


print("\n📋 Creating Customer Dataset...")

df_cust = data['customers'].copy().merge(data['customer_types'], on='CustomerTypeID', how='left')

# Account summary per customer
acc_summary = data['accounts'].groupby('CustomerID').agg(
    AccountCount=('AccountID', 'count'),
    TotalBalance=('Balance', 'sum'),
    AvgBalance=('Balance', 'mean'),
    MaxBalance=('Balance', 'max'),
    FirstAccountDate=('OpeningDate', 'min'),
    LastAccountDate=('OpeningDate', 'max')
).reset_index()

df_cust = df_cust.merge(acc_summary, on='CustomerID', how='left')
for col in ['AccountCount', 'TotalBalance', 'AvgBalance', 'MaxBalance']:
    df_cust[col] = df_cust[col].fillna(0)

# Loan summary per customer
cust_loans = data['loans'].merge(data['accounts'][['AccountID', 'CustomerID']], on='AccountID')
loan_summary_cust = cust_loans.groupby('CustomerID').agg(
    CustomerLoanCount=('LoanID', 'count'),
    CustomerTotalPrincipal=('PrincipalAmount', 'sum'),
    CustomerAvgPrincipal=('PrincipalAmount', 'mean'),
    CustomerMaxPrincipal=('PrincipalAmount', 'max'),
    CustomerAvgInterestRate=('InterestRate', 'mean'),
    CustomerMaxInterestRate=('InterestRate', 'max')
).reset_index()

df_cust = df_cust.merge(loan_summary_cust, on='CustomerID', how='left')
for col in ['CustomerLoanCount', 'CustomerTotalPrincipal', 'CustomerAvgPrincipal', 
            'CustomerMaxPrincipal', 'CustomerAvgInterestRate', 'CustomerMaxInterestRate']:
    df_cust[col] = df_cust[col].fillna(0)

print(f"   ✅ {df_cust.shape[0]:,} rows × {df_cust.shape[1]} columns")

# SAVE

print("\n💾 Saving datasets...")
eda_path = Path('../data/eda/')
eda_path.mkdir(parents=True, exist_ok=True)

df_trans.to_csv(eda_path / 'transactions_unified.csv', index=False)
df_acc.to_csv(eda_path / 'accounts_unified.csv', index=False)
df_cust.to_csv(eda_path / 'customers_unified.csv', index=False)

print(f"   ✅ Saved to {eda_path}")


# SUMMARY


print("\n📊 SUMMARY:")
print(f"   Transactions: {df_trans.shape[0]:,} rows × {df_trans.shape[1]} cols")
print(f"   Accounts:     {df_acc.shape[0]:,} rows × {df_acc.shape[1]} cols")
print(f"   Customers:    {df_cust.shape[0]:,} rows × {df_cust.shape[1]} cols")

print("\n✅ DONE!")


📊 CREATING UNIFIED DATASET FOR EDA

✅ Loaded 11 tables

📋 Creating Transaction Dataset...
   ✅ 48,510 rows × 35 columns

📋 Creating Account Dataset...
   ✅ 1,651 rows × 24 columns

📋 Creating Customer Dataset...
   ✅ 1,100 rows × 19 columns

💾 Saving datasets...
   ✅ Saved to ..\data\eda

📊 SUMMARY:
   Transactions: 48,510 rows × 35 cols
   Accounts:     1,651 rows × 24 cols
   Customers:    1,100 rows × 19 cols

✅ DONE!
